In [ ]:
import os
import string
import math
import re
from nltk.corpus import stopwords
from collections import defaultdict
from nltk.stem import PorterStemmer
import nltk

# Download stopwords only
nltk.download('stopwords')

def simple_tokenize(text):
    tokens = re.findall(r'\b\w+\b', text.lower())
    return tokens

def preprocess_text(text):
    cleaned_text = text.translate(str.maketrans('', '', string.punctuation)).lower()
    tokens = simple_tokenize(cleaned_text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [token for token in tokens if token not in stop_words]
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]
    return stemmed_tokens

def calculate_tfidf(documents):
    tf = defaultdict(lambda: defaultdict(int))
    idf = defaultdict(int)
    num_docs = len(documents)

    for doc_id, tokens in documents.items():
        for token in tokens:
            tf[doc_id][token] += 1
        for token in set(tokens):
            idf[token] += 1

    tfidf = defaultdict(lambda: defaultdict(float))
    for doc_id, token_freqs in tf.items():
        doc_len = sum(token_freqs.values())
        for token, freq in token_freqs.items():
            tfidf[doc_id][token] = (freq / doc_len) * math.log(num_docs / (1 + idf[token]))
    return tfidf, idf, num_docs

def vectorize(tokens, idf, num_docs):
    tf = defaultdict(int)
    for token in tokens:
        tf[token] += 1
    doc_len = sum(tf.values())
    tfidf_vec = {}
    for token, freq in tf.items():
        tfidf_vec[token] = (freq / doc_len) * math.log(num_docs / (1 + idf.get(token, 0)))
    return tfidf_vec

def cosine_similarity(vec1, vec2):
    # Compute cosine similarity between two sparse TF-IDF vectors
    common_tokens = set(vec1.keys()) & set(vec2.keys())
    numerator = sum(vec1[t] * vec2[t] for t in common_tokens)
    sum1 = sum(v**2 for v in vec1.values())
    sum2 = sum(v**2 for v in vec2.values())
    denominator = math.sqrt(sum1) * math.sqrt(sum2)
    if denominator == 0:
        return 0.0
    else:
        return numerator / denominator

def process_files(folder_path):
    documents = {}
    raw_texts = {}
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            with open(os.path.join(folder_path, filename), 'r') as file:
                text = file.read()
                preprocessed_text = preprocess_text(text)
                doc_id = filename.split('.')[0]
                documents[doc_id] = preprocessed_text
                raw_texts[doc_id] = text
    return documents, raw_texts

# Main: load docs, preprocess, calculate TF-IDF
folder_path = '/Bank'  # Update your path here
documents, raw_texts = process_files(folder_path)
tfidf_scores, idf, num_docs = calculate_tfidf(documents)

# Chatbot query function
def chatbot_query(query, top_k=3):
    query_tokens = preprocess_text(query)
    query_vec = vectorize(query_tokens, idf, num_docs)

    # Calculate similarity with all documents
    scores = []
    for doc_id, doc_vec in tfidf_scores.items():
        sim = cosine_similarity(query_vec, doc_vec)
        scores.append((doc_id, sim))
    scores.sort(key=lambda x: x[1], reverse=True)

    # Return top_k most relevant documents with snippets
    results = []
    for doc_id, score in scores[:top_k]:
        results.append({
            'doc_id': doc_id,
            'similarity_score': score,
            'content_snippet': raw_texts[doc_id][:300]  # first 300 chars as snippet
        })
    return results




In [11]:
# Instead of fixed query, get input from user
query = input("Enter your query: ")

results = chatbot_query(query)

print(f"Query: {query}")
for res in results:
    print(f"Document: {res['doc_id']}, Score: {res['similarity_score']:.4f}")
    print(f"Snippet: {res['content_snippet']}\n{'-'*50}")


Enter your query: what is loan?
Query: what is loan?
Document: 17, Score: 0.5464
Snippet: Loan interest rate
The interest rate in Loan account start from 14 percent. There are different offering in loann account as per opening additional fixed deposit account in it.
The interest rate on a loan account can vary widely depending on several factors, including the type of loan, the lender, p
--------------------------------------------------
Document: 12, Score: 0.5135
Snippet: open loan account process
To open a loan account, various documents are typically required by the lending institution. 

The specific documents needed can vary depending on the type of loan, the policies of the lending institution, and local regulations. However, common documents requested when open
--------------------------------------------------
Document: 19, Score: 0.2700
Snippet: Bank services
Banks in Nepal offer a range of financial products and services to cater to the needs of individuals, 
businesses, and

In [13]:
# Instead of fixed query, get input from user
query = input("Enter your query: ")

results = chatbot_query(query)

print(f"Query: {query}")
for res in results:
    print(f"Document: {res['doc_id']}, Score: {res['similarity_score']:.4f}")
    print(f"Snippet: {res['content_snippet']}\n{'-'*50}")


Enter your query: what is Bank ?
Query: what is Bank ?
Document: 20, Score: 0.0000
Snippet: Bill payment system
Bill payment services offered by banks enable customers to conveniently pay various bills without 
the need to visit individual service providers. Here are common types of bill payments facilitated 
by banks:
Utility Bills: 
Credit Card Bills: 
Internet and Cable TV Bills: 
Insur
--------------------------------------------------
Document: 13, Score: 0.0000
Snippet: HR application
When applying to the Human Resources (HR) department of a bank or any organization, several documents are typically required to support your application. Here's a list of common documents you might need:

Resume/Curriculum Vitae (CV):

This is a detailed summary of your education, wor
--------------------------------------------------
Document: 17, Score: 0.0000
Snippet: Loan interest rate
The interest rate in Loan account start from 14 percent. There are different offering in loann account as per o